# Ejercicio 10: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

**Estudiante:** Kevin Alvear

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [1]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

C:\Users\Kevin Alvear\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\beir\util.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

../data/beir_datasets\scifact.zip: 100%|██████████| 2.69M/2.69M [00:11<00:00, 239kiB/s] 


'../data/beir_datasets\\scifact'

In [3]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

  0%|          | 0/5183 [00:00<?, ?it/s]

100%|██████████| 5183/5183 [00:00<00:00, 54381.55it/s]


In [4]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [5]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [6]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [7]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [9]:
from collections import defaultdict

from beir.retrieval.evaluation import EvaluateRetrieval
from rank_bm25 import BM25Okapi

def tokenize(text):
    return text.lower().split()

# Construimos el índice BM25 sobre el corpus completo.
doc_ids = list(corpus.keys())
tokenized_corpus = [
    tokenize((doc.get("title", "") + " " + doc.get("text", "")).strip())
    for doc in corpus.values()
 ]
bm25 = BM25Okapi(tokenized_corpus)

# Recuperamos candidatos para cada query.
results = {}
top_k = 100
for qid, query in queries.items():
    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)
    top_indices = scores.argsort()[::-1][:top_k]
    results[qid] = {doc_ids[idx]: float(scores[idx]) for idx in top_indices}

# Evaluación del baseline.
evaluator = EvaluateRetrieval()
ndcg, _map, recall, precision = evaluator.evaluate(qrels, results, [10])

print("BM25 baseline metrics @10")
print(f"nDCG@10: {ndcg['NDCG@10']:.4f}")
print(f"Recall@10: {recall['Recall@10']:.4f}")

BM25 baseline metrics @10
nDCG@10: 0.5597
Recall@10: 0.6862


## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [12]:
import pandas as pd

def get_doc_text(doc_id, max_chars=300):
    doc = corpus[doc_id]
    text = (doc.get("title", "") + " " + doc.get("text", "")).strip()
    return text[:max_chars]

MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-2-v2"
MAX_CANDIDATES = 20
MAX_DOC_CHARS = 300
BATCH_SIZE = 32

if 'cross_encoder' not in globals() or getattr(cross_encoder, 'model_name', None) != MODEL_NAME:
    cross_encoder = CrossEncoder(MODEL_NAME)

reranked_results = {}
for query_id, query_text in queries.items():
    candidate_ids = list(results[query_id].keys())[:MAX_CANDIDATES]
    pairs = [(query_text, get_doc_text(doc_id, max_chars=MAX_DOC_CHARS)) for doc_id in candidate_ids]
    ce_scores = cross_encoder.predict(pairs, batch_size=BATCH_SIZE, show_progress_bar=False)
    reranked_order = sorted(zip(candidate_ids, ce_scores), key=lambda item: item[1], reverse=True)
    reranked_results[query_id] = {doc_id: float(score) for doc_id, score in reranked_order}

baseline_top10 = list(results[qid].keys())[:10]
reranked_top10 = list(reranked_results[qid].keys())[:10]

comparison_rows = []
for doc_id in sorted(set(baseline_top10) | set(reranked_top10)):
    baseline_rank = baseline_top10.index(doc_id) + 1 if doc_id in baseline_top10 else None
    reranked_rank = reranked_top10.index(doc_id) + 1 if doc_id in reranked_top10 else None
    if baseline_rank != reranked_rank:
        comparison_rows.append({
            "doc_id": doc_id,
            "baseline_rank": baseline_rank,
            "reranked_rank": reranked_rank,
            "title": corpus[doc_id].get("title", "")[:120]
        })

changed_top10 = pd.DataFrame(comparison_rows).sort_values(["reranked_rank", "baseline_rank"], na_position="last") if comparison_rows else pd.DataFrame(columns=["doc_id", "baseline_rank", "reranked_rank", "title"])

print("Top 10 BM25:")
print(baseline_top10)
print("\nTop 10 re-ranked con cross-encoder:")
print(reranked_top10)
print("\nDocumentos que cambian de posición en el top 10:")
display(changed_top10)

Loading weights: 100%|██████████| 41/41 [00:00<00:00, 8579.88it/s]


Top 10 BM25:
['928281', '19343151', '10698739', '4387484', '13923069', '1084345', '38793927', '16630060', '18218379', '26008462']

Top 10 re-ranked con cross-encoder:
['18218379', '19343151', '3113630', '13923069', '26008462', '16627684', '15435343', '3419802', '16630060', '4336849']

Documentos que cambian de posición en el top 10:


,doc_id,baseline_rank,reranked_rank,title
6,18218379,9.0,1.0,Quantitative analysis of tumor-derived methyla...
8,3113630,NaN,3.0,Nuclear accumulation of HDAC4 in ATM deficienc...
2,13923069,5.0,4.0,Targeted nanoparticles containing the proresol...
7,26008462,10.0,5.0,Tissue transglutaminase (TG2)--a wound respons...
4,16627684,NaN,6.0,Hmga2 Promotes Neural Stem Cell Self-Renewal i...
3,15435343,NaN,7.0,The inflammasome component NLRP3 impairs antit...
9,3419802,NaN,8.0,Microenvironment-dependent growth of pre-neopl...
5,16630060,8.0,9.0,Genotoxic Stress Abrogates Renewal of Melanocy...
11,4336849,NaN,10.0,Chloroquine resistance not linked to mdr-like ...
13,928281,1.0,NaN,Failure of cell cleavage induces senescence in...


## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [16]:
import re
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def tokenize_light(text):
    return re.findall(r"\w+", text.lower())

query_tokens_cache = {query_id: tokenize_light(query_text) for query_id, query_text in queries.items()}
doc_cache = {}
for doc_id, doc in corpus.items():
    doc_text = (doc.get("title", "") + " " + doc.get("text", "")).strip()
    doc_tokens = tokenize_light(doc_text)
    doc_cache[doc_id] = {
        "bm25": 0.0,
        "len_doc": len(doc_tokens),
        "token_set": set(doc_tokens),
    }

def build_ltr_features(query_id, doc_id):
    query_tokens = query_tokens_cache[query_id]
    doc_token_set = doc_cache[doc_id]["token_set"]
    overlap = sum(1 for token in query_tokens if token in doc_token_set)
    return [
        results[query_id].get(doc_id, 0.0),
        len(query_tokens),
        doc_cache[doc_id]["len_doc"],
        overlap,
        overlap / max(1, len(query_tokens)),
        overlap / max(1, len(doc_token_set)),
    ]

MAX_CANDIDATES_LTR = 20
train_query_ids = list(queries.keys())
X_train = []
y_train = []

for query_id in train_query_ids:
    candidate_ids = list(results[query_id].keys())[:MAX_CANDIDATES_LTR]
    relevant_docs = set(qrels.get(query_id, {}).keys())
    for doc_id in candidate_ids:
        X_train.append(build_ltr_features(query_id, doc_id))
        y_train.append(1 if doc_id in relevant_docs else 0)

ltr_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
)
ltr_model.fit(X_train, y_train)

reranked_results_ltr = {}
for query_id in queries:
    candidate_ids = list(results[query_id].keys())[:MAX_CANDIDATES_LTR]
    feature_rows = [build_ltr_features(query_id, doc_id) for doc_id in candidate_ids]
    scores = ltr_model.predict_proba(feature_rows)[:, 1]
    reranked_order = sorted(zip(candidate_ids, scores), key=lambda item: item[1], reverse=True)
    reranked_results_ltr[query_id] = {doc_id: float(score) for doc_id, score in reranked_order}

baseline_top10 = list(results[qid].keys())[:10]
ltr_top10 = list(reranked_results_ltr[qid].keys())[:10]

comparison_rows = []
for doc_id in sorted(set(baseline_top10) | set(ltr_top10)):
    baseline_rank = baseline_top10.index(doc_id) + 1 if doc_id in baseline_top10 else None
    ltr_rank = ltr_top10.index(doc_id) + 1 if doc_id in ltr_top10 else None
    if baseline_rank != ltr_rank:
        comparison_rows.append({
            "doc_id": doc_id,
            "baseline_rank": baseline_rank,
            "ltr_rank": ltr_rank,
            "title": corpus[doc_id].get("title", "")[:120]
        })

changed_top10 = pd.DataFrame(comparison_rows).sort_values(["ltr_rank", "baseline_rank"], na_position="last") if comparison_rows else pd.DataFrame(columns=["doc_id", "baseline_rank", "ltr_rank", "title"])

print("Top 10 BM25:")
print(baseline_top10)
print("\nTop 10 re-ranked con LTR:")
print(ltr_top10)
print("\nDocumentos que cambian de posición en el top 10:")
display(changed_top10)

Top 10 BM25:
['928281', '19343151', '10698739', '4387484', '13923069', '1084345', '38793927', '16630060', '18218379', '26008462']

Top 10 re-ranked con LTR:
['10698739', '13923069', '928281', '19343151', '4387484', '1084345', '7114092', '38793927', '11569583', '16630060']

Documentos que cambian de posición en el top 10:


,doc_id,baseline_rank,ltr_rank,title
0,10698739,3.0,1.0,Modulation of mitochondrial function and morph...
2,13923069,5.0,2.0,Targeted nanoparticles containing the proresol...
10,928281,1.0,3.0,Failure of cell cleavage induces senescence in...
5,19343151,2.0,4.0,p16INK4A is a robust in vivo biomarker of cell...
8,4387484,4.0,5.0,G-protein-coupled receptor of Kaposi's sarcoma...
9,7114092,NaN,7.0,From bloodjournal.hematologylibrary.org at PEN...
7,38793927,7.0,8.0,Osteoclast nuclei of myeloma patients show chr...
1,11569583,NaN,9.0,Deregulated DNA polymerase β strengthens ioniz...
3,16630060,8.0,10.0,Genotoxic Stress Abrogates Renewal of Melanocy...
4,18218379,9.0,NaN,Quantitative analysis of tumor-derived methyla...


## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10

In [17]:
from beir.retrieval.evaluation import EvaluateRetrieval

if 'reranked_results_ltr' in globals():
    eval_results = reranked_results_ltr
elif 'reranked_results' in globals():
    eval_results = reranked_results
else:
    raise NameError("No reranked results found. Run the reranking cell first.")

evaluator = EvaluateRetrieval()
ndcg, _map, recall, precision = evaluator.evaluate(qrels, eval_results, [10])

print("Post re-ranking metrics @10")
print(f"nDCG@10: {ndcg['NDCG@10']:.4f}")
print(f"MAP@10: {_map['MAP@10']:.4f}")
print(f"Recall@10: {recall['Recall@10']:.4f}")

Post re-ranking metrics @10
nDCG@10: 0.5707
MAP@10: 0.5316
Recall@10: 0.6772
